In [1]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

2026-02-01 18:00:36.385870: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-01 18:00:36.434989: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-01 18:00:37.771649: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
sentences=[
    'I like eggs and cheese.','I love chocolates and bunnies.', 'I hate onions.'
]

In [3]:
MAX_VOCAB_SIZE=20000

In [4]:
vectorization_layer=TextVectorization(max_tokens=MAX_VOCAB_SIZE)

I0000 00:00:1769950839.163320 2947631 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3907 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1660 SUPER, pci bus id: 0000:01:00.0, compute capability: 7.5


In [5]:
vectorization_layer.adapt(sentences)

In [6]:
sequences=vectorization_layer(sentences)
print(sequences)

tf.Tensor(
[[ 2  6  8  3 10]
 [ 2  5  9  3 11]
 [ 2  7  4  0  0]], shape=(3, 5), dtype=int64)


In [7]:
vectorization_layer.get_vocabulary()

['',
 '[UNK]',
 np.str_('i'),
 np.str_('and'),
 np.str_('onions'),
 np.str_('love'),
 np.str_('like'),
 np.str_('hate'),
 np.str_('eggs'),
 np.str_('chocolates'),
 np.str_('cheese'),
 np.str_('bunnies')]

In [8]:
# How to get the word to index mapping?
# It's not included, but it's redundant since it can be derived
word2idx = {v: k for k, v in enumerate(vectorization_layer.get_vocabulary())}
print(word2idx)

{'': 0, '[UNK]': 1, np.str_('i'): 2, np.str_('and'): 3, np.str_('onions'): 4, np.str_('love'): 5, np.str_('like'): 6, np.str_('hate'): 7, np.str_('eggs'): 8, np.str_('chocolates'): 9, np.str_('cheese'): 10, np.str_('bunnies'): 11}


In [9]:
vectorization_layer_truncated=TextVectorization(max_tokens=MAX_VOCAB_SIZE,output_sequence_length=3)

In [10]:
vectorization_layer_truncated.adapt(sentences)
sequences_truncated=vectorization_layer_truncated(sentences)
print(sequences_truncated)

tf.Tensor(
[[2 6 8]
 [2 5 9]
 [2 7 4]], shape=(3, 3), dtype=int64)


In [11]:
# ragged (no padding) (TF backend only)
vectorization_layer_ragged = TextVectorization(
    max_tokens=MAX_VOCAB_SIZE,
    ragged=True,
)

# fit
vectorization_layer_ragged.adapt(sentences)

# predict
ragged_sequences = vectorization_layer_ragged(sentences)
print(ragged_sequences)

<tf.RaggedTensor [[2, 6, 8, 3, 10], [2, 5, 9, 3, 11], [2, 7, 4]]>


In [12]:
# pad at front instead of back
# not supported in TextVectorization layer itself
from tensorflow.keras.utils import pad_sequences

# defaults:
# tf.keras.utils.pad_sequences(
#     sequences,
#     maxlen=None,
#     dtype='int32',
#     padding='pre',
#     truncating='pre',
#     value=0.0
# )

padded = pad_sequences(ragged_sequences.to_list())
print(padded)

[[ 2  6  8  3 10]
 [ 2  5  9  3 11]
 [ 0  0  2  7  4]]
